In [ ]:
import os

import boto3
import polars as pl
from boto3.s3.transfer import TransferConfig
from dotenv import load_dotenv

load_dotenv()

In [ ]:
def progress_callback(bytes_transferred: int) -> None:
    print(f"\rUploaded: {bytes_transferred / (1024*1024):.1f} MB", end="")

In [ ]:
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["R2_ENDPOINT"],
    aws_access_key_id=os.environ["R2_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["R2_SECRET_ACCESS_KEY"],
    region_name="auto",
)

config = TransferConfig(
    multipart_threshold=1024 * 25,
    multipart_chunksize=1024 * 25,
    max_concurrency=4,
    use_threads=True,
)

In [ ]:
storage_options = {
    "aws_access_key_id": os.environ["R2_ACCESS_KEY_ID"],
    "aws_secret_access_key": os.environ["R2_SECRET_ACCESS_KEY"],
    "endpoint_url": os.environ["R2_ENDPOINT"],
    "aws_region": "auto",
}

df = pl.scan_parquet(
    f"s3://{os.environ['R2_BUCKET']}/raw/btc_usdt_lob_snapshot_2022-10-20.parquet",
    storage_options=storage_options,
)
print(df.collect().head())
print(df.collect().shape)

In [ ]:
print(df.select(pl.col("asks").list.len().alias("ask_levels")).describe())
print(df.select(pl.col("bids").list.len().alias("bid_levels")).describe())

In [ ]:
# check if first few entries are sorted ascending (asks) / descending (bids) by price
sample = df.select("asks", "bids").head(3).collect()

for row in sample.iter_rows(named=True):
    ask_prices = (
        [x["price"] for x in row["asks"][:5]]
        if "price" in row["asks"][0]
        else row["asks"][:5]
    )
    bid_prices = (
        [x["price"] for x in row["bids"][:5]]
        if "price" in row["bids"][0]
        else row["bids"][:5]
    )
    print("first 5 ask prices:", ask_prices)
    print("first 5 bid prices:", bid_prices)
    print()

In [ ]:
print(df.schema)

In [ ]:
n = 10

df_lazy = pl.scan_parquet(
    f"s3://{os.environ['R2_BUCKET']}/raw/btc_usdt_lob_snapshot_2022-10-20.parquet",
    storage_options=storage_options,
)

truncated = df_lazy.select(
    "time",
    pl.col("asks").list.head(n).alias("asks"),
    pl.col("bids").list.head(n).alias("bids"),
)

exprs: list[str | pl.Expr] = ["time"]
for i in range(n):
    exprs.append(
        pl.col("asks").list.get(i).struct.field("price").alias(f"ask_price_{i+1}")
    )
    exprs.append(
        pl.col("asks").list.get(i).struct.field("size").alias(f"ask_size_{i+1}")
    )
    exprs.append(
        pl.col("bids").list.get(i).struct.field("price").alias(f"bid_price_{i+1}")
    )
    exprs.append(
        pl.col("bids").list.get(i).struct.field("size").alias(f"bid_size_{i+1}")
    )

flat = truncated.select(exprs).collect()
print(flat.shape)
print(flat.head())

In [ ]:
null_counts = flat.null_count()
print(null_counts)

In [ ]:
flat.write_parquet("../data/lob_snapshots_btc_usdt_2022-10-20.parquet")

In [ ]:
s3.upload_file(
    "../data/lob_snapshots_btc_usdt_2022-10-20.parquet",
    os.environ["R2_BUCKET"],
    "lob_snapshots/btc_usdt/2022-10-20.parquet",
    Config=config,
    Callback=progress_callback,
)
print("\ndone")